# Agriculture Data Cleaning: USDA-NASS Chemical / Fertilizer / Feed Spending

Cleans the USDA-NASS Quick Stats **production-expense** export (Iowa,
state-level) into a tidy table: one row per `(year, expense_category, unit)`.

**Input:**  `data/tabular/01_raw/agriculture/Chemical-Fertilizer-Feed-Spending.csv`
**Output:** `data/tabular/02_clean/agriculture/chemical-fertilizer-feed-spending-clean.csv`

**What's in it** — annual statewide farm spending (`SURVEY` program) on three
input categories — `CHEMICAL TOTALS`, `FERTILIZER TOTALS, INCL LIME & SOIL
CONDITIONERS`, and `FEED` — each reported four ways via the `Data Item` unit:
total `$`, `$ / OPERATION`, `PCT OF OPERATIONS` reporting the expense, and
`PCT OF PRODUCTION EXPENSES`. We split the category from the unit so each
measure is a clean, filterable column pair.

**Pipeline**
1. **Load** as strings. 2. **Drop** empty/constant columns (state-level: all geo
columns blank; `Domain`, `CV (%)` blank/constant). 3. **Parse** `year` and
`value`. 4. **Unpack** `Data Item` into `expense_category` + `unit`. 5. **Key**,
check, save.

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "agriculture"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "agriculture"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

In [ ]:
# --- USDA-NASS shared cleaning helpers ------------------------------------
#
# NASS uses parenthetical letter codes in place of numbers. They are NOT data;
# they encode *why* a number is absent, so they must become NaN (never 0):
#   (D) withheld to avoid disclosing data for individual operations
#   (Z) value rounds to less than half the unit shown
#   (X) not applicable
#   (NA) not available
#   (H) sampling CV >= 99.95% (estimate too unreliable to publish)
#   (L) sampling CV  <  0.05%
# (D) appears in `Value`; (D)/(H)/(L) appear in `CV (%)`.
NASS_SUPPRESSION = {"(D)", "(Z)", "(X)", "(NA)", "(H)", "(L)", "(NA)", "(S)"}


def parse_nass_numeric(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    """Parse a NASS Value/CV column into (float, was_suppressed_flag).

    Strips thousands separators, maps every suppression code to NaN, and flags
    which rows were a real suppression code (vs. genuinely blank) so downstream
    users can tell "censored" apart from "not collected".
    """
    s = series.astype("string").str.strip()
    suppressed = s.isin(NASS_SUPPRESSION)
    cleaned = s.mask(s.isin(NASS_SUPPRESSION))           # codes -> <NA>
    cleaned = cleaned.str.replace(",", "", regex=False)  # 1,234 -> 1234
    return pd.to_numeric(cleaned, errors="coerce"), suppressed.fillna(False)


# A Data Item is "<COMMODITY DETAIL> - <STATISTIC>[, MEASURED IN <UNIT>]",
# e.g. "CORN, GRAIN - YIELD, MEASURED IN BU / ACRE" or "CORN - ACRES PLANTED".
_DATA_ITEM_RE = re.compile(r"^(?P<detail>.+?) - (?P<stat>.+?)(?:, MEASURED IN (?P<unit>.+))?$")


def parse_data_item(item: str) -> tuple[str, str, str | None]:
    """Split a Data Item string into (commodity_detail, statistic, unit)."""
    m = _DATA_ITEM_RE.match(item)
    if not m:
        raise ValueError(f"Unparseable Data Item: {item!r}")
    return m.group("detail"), m.group("stat"), m.group("unit")

## Step 1 - Load

In [ ]:
RAW_FILE = "Chemical-Fertilizer-Feed-Spending.csv"
raw = pd.read_csv(RAW_DIR / RAW_FILE, dtype="string")
n_raw = len(raw)
print(f"Loaded {n_raw:,} rows x {raw.shape[1]} cols from {RAW_FILE}")
raw.head()

In [ ]:
# Every Data Item must parse, and every State ANSI must be Iowa (19) -- guard
# against a future re-pull silently changing the schema or scope.
assert raw["State ANSI"].dropna().eq("19").all(), "Non-Iowa rows present!"
_unparsed = [it for it in raw["Data Item"].dropna().unique() if not _DATA_ITEM_RE.match(it)]
assert not _unparsed, f"Unparseable Data Items: {_unparsed}"
print("Schema guards passed: all Iowa, all Data Items parse.")

## Step 2 - Drop structurally-empty and constant columns

State-level again: all county/ag-district/watershed columns are blank, as is
`CV (%)`. `Domain` / `Domain Category` are constant (`TOTAL` / `NOT SPECIFIED`)
and carry no breakdown here, so they go too.

In [ ]:
EMPTY_COLS = ["Week Ending", "Ag District", "Ag District Code", "County",
              "County ANSI", "Zip Code", "Region", "Watershed", "CV (%)"]
CONST_COLS = ["Program", "Period", "Geo Level", "State",
              "Domain", "Domain Category", "watershed_code"]

for col in EMPTY_COLS:
    assert raw[col].isna().all(), f"Expected {col!r} empty but it has values"
for col in CONST_COLS:
    assert raw[col].nunique(dropna=True) <= 1, f"Expected {col!r} constant: {raw[col].unique()}"

df = raw.drop(columns=EMPTY_COLS + CONST_COLS)
print(f"Dropped {len(EMPTY_COLS)} empty + {len(CONST_COLS)} constant columns; "
      f"{df.shape[1]} columns remain")

## Step 3 - Parse `year`, `state_fips`, and `value`

`Value` here is fully numeric (no suppression codes in this small table), but we
use the same NASS-aware parser for robustness against future re-pulls.

In [ ]:
df["year"] = df["Year"].astype(int)
df["state_fips"] = df["State ANSI"].str.zfill(2)
df["value"], df["value_suppressed"] = parse_nass_numeric(df["Value"])
print(f"value: {df['value'].notna().sum():,} numeric, "
      f"{df['value_suppressed'].sum():,} suppressed")

## Step 4 - Unpack `Data Item` into category + unit

`Data Item` is `<EXPENSE CATEGORY> - EXPENSE, MEASURED IN <UNIT>`. The detail is
the expense category, the statistic is always `EXPENSE`, and the unit
distinguishes the four ways each category is reported.

In [ ]:
parsed = df["Data Item"].map(parse_data_item)
df["expense_category"] = parsed.map(lambda t: t[0])
df["statistic"] = parsed.map(lambda t: t[1])    # EXPENSE
df["unit"] = parsed.map(lambda t: t[2])

df = df.rename(columns={"Commodity": "commodity"})
print(df[["commodity", "expense_category", "unit"]].drop_duplicates()
        .sort_values(["expense_category", "unit"]).to_string(index=False))

## Step 5 - Select, order, and enforce a unique key

In [ ]:
OUTPUT_COLS = [
    "year", "state_fips", "commodity", "expense_category", "statistic", "unit",
    "value", "value_suppressed",
]
clean = df[OUTPUT_COLS].sort_values(
    ["year", "expense_category", "unit"]
).reset_index(drop=True)

KEY = ["year", "expense_category", "unit"]
dupes = clean.duplicated(KEY).sum()
assert dupes == 0, f"{dupes} duplicate key rows!"
print(f"Key is unique across {len(clean):,} rows.")
clean.head()

## Step 6 - Sanity check

In [ ]:
print(f"Rows: {len(clean):,}  |  years: {clean['year'].min()}-{clean['year'].max()}")
print(f"categories: {clean['expense_category'].unique().tolist()}\n")
print("Total $ spend per category per year (unit == '$'):")
dollars = clean[clean.unit == "$"].pivot_table(
    index="year", columns="expense_category", values="value")
print(dollars.round(0).to_string())

## Step 7 - Save

In [ ]:
out_file = CLEAN_DIR / "chemical-fertilizer-feed-spending-clean.csv"
clean.to_csv(out_file, index=False)
print(f"Saved {len(clean):,} rows -> {out_file}")